# Dia 4: modelação

Responde às perguntas que o Dia 3 deixou em aberto, porque agora **controlamos várias variáveis ao mesmo tempo**.

| Secção | Pergunta | Método |
|---|---|---|
| 1 | Q1: a adoção de IA subiu, mesmo controlando a composição? | Regressão logística + padronização |
| 2 | Q2: quem usa IA ganha mais? Os salários subiram? | Regressão linear (OLS) com controlos crescentes |
| 3 | Q3: que fatores se associam ao salário? | XGBoost + SHAP |
| 4 | Q4: a confiança na IA mudou? | Mann-Whitney + tamanho de efeito |
| 5 | Q5: que perfis de developers existem? | K-Means |

**Regra de leitura:** tudo isto são **associações**, não efeitos causais.

In [23]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.models_moduls.features import load_clean, build_features, cols_of, exp_band, ORDEM_EXP
from src.models_moduls import modeling as M
from src.eda_utils import save_fig

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
os.makedirs("reports", exist_ok=True); os.makedirs("models", exist_ok=True)

df = load_clean()
print(df.shape)
df.groupby("year").agg(n=("comp_yearly", "size"), mediana=("comp_yearly", "median"))

(39182, 38)


,n,mediana
year,,
2024,20669,66592.0
2025,18513,81210.0


In [24]:
# Matriz de variáveis: só linhas com uses_ai conhecido (as restantes não entram nos modelos de IA)
X, y, groups, d = build_features(df)
print(X.shape, "| linhas:", len(d), "| por ano:", d["year"].value_counts().to_dict())
pd.Series(groups).value_counts().rename("n_colunas").to_frame()

(38706, 86) | linhas: 38706 | por ano: {2024: 20669, 2025: 18037}


,n_colunas
pais,31
linguagem,20
funcao,10
org,9
educacao,6
remoto,6
experiencia,2
ia,1
ano,1


## 1. Q1: a adoção subiu, mesmo controlando a composição?
A amostra de 2025 é mais sénior e tem outra mistura de países. A regressão logística controla isso, e a **padronização** responde: *qual seria a subida se a composição fosse a mesma nos dois anos?*

In [25]:
q1 = M.adoption_adjusted(X, groups, d)
q1_tab = pd.DataFrame({"valor": {
    "subida bruta (pontos)": round(q1["bruta_pp"], 1),
    "subida ajustada (pontos)": round(q1["ajustada_pp"], 1),
    "odds ratio de 2025": round(q1["odds_ratio"], 2),
    "IC 95% do odds ratio": f"{q1['or_lo']:.2f} a {q1['or_hi']:.2f}",
    "n": q1["n"]}})
q1_tab.to_csv("reports/q1_adocao_ajustada.csv")
q1_tab

,valor
subida bruta (pontos),18.1
subida ajustada (pontos),20.7
odds ratio de 2025,3.02
IC 95% do odds ratio,2.80 a 3.25
n,38706


In [26]:
fig = px.bar(x=["Bruta", "Ajustada"], y=[q1["bruta_pp"], q1["ajustada_pp"]],
             text=[f"{q1['bruta_pp']:.1f}", f"{q1['ajustada_pp']:.1f}"],
             title="Subida da adoção de IA 2024→2025 (pontos percentuais)",
             labels={"x": "", "y": "pontos percentuais"})
fig.update_traces(textposition="outside")
fig.show(); save_fig(fig, "g11_adocao_bruta_vs_ajustada")

WindowsPath('figures/g11_adocao_bruta_vs_ajustada.html')

## 2. Q2: quem usa IA ganha mais? Os salários subiram?
Regressão linear de `log(salário)` com controlos crescentes. Como a variável está em logaritmos, o coeficiente lê-se como **variação percentual**.

Cada linha acrescenta controlos. Observa como o efeito de `uses_ai` **e** o de `year_2025` mudam. A segunda resolve a dúvida do Dia 3: *o aumento da mediana é real ou é composição?*

In [27]:
ols = M.ols_specs(X, y, groups)
ols.to_csv("reports/q2_ols.csv", index=False)
ols

,modelo,n,r2,ia_pct,ia_lo,ia_hi,ano_pct,ano_lo,ano_hi
0,1. só IA + ano,38706,0.0,-16.5,-18.2,-14.8,27.2,24.8,29.7
1,2. + experiência,38706,0.2,-7.4,-9.1,-5.6,9.5,7.6,11.5
2,3. + país,38706,0.5,4.0,2.5,5.4,4.7,3.3,6.2
3,"4. + função, linguagens, educação...",38706,0.6,3.1,1.7,4.5,0.6,-1.3,2.6


In [28]:
fig = go.Figure()
for col, nome in (("ia", "Usar IA"), ("ano", "Ano 2025 (vs 2024)")):
    fig.add_trace(go.Scatter(
        x=ols["modelo"], y=ols[f"{col}_pct"], mode="markers+lines", name=nome,
        error_y=dict(type="data", symmetric=False,
                     array=ols[f"{col}_hi"] - ols[f"{col}_pct"],
                     arrayminus=ols[f"{col}_pct"] - ols[f"{col}_lo"])))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(title="Efeito estimado no salário (%) à medida que se acrescentam controlos",
                  yaxis_title="variação % no salário", xaxis_title="")
fig.show(); save_fig(fig, "g12_ols_coeficientes")

WindowsPath('figures/g12_ols_coeficientes.html')

### Efeito de usar IA dentro de cada faixa de experiência (modelo completo)
No Dia 3 o sinal mudava com a experiência. Aqui vê-se o efeito **depois de controlar país, função e linguagens**.

In [29]:
by_band = M.effect_by_band(X, y, groups, d)
by_band.to_csv("reports/q2_efeito_por_faixa.csv", index=False)
by_band

,faixa,n,pct,lo,hi
0,0-1 anos,1807,7.1,-0.9,15.7
1,2-4 anos,7095,-1.2,-4.7,2.4
2,5-9 anos,9977,3.1,0.4,6.0
3,10-19 anos,11964,5.6,3.2,8.1
4,20+ anos,7863,8.3,5.7,10.9


In [30]:
fig = go.Figure(go.Scatter(
    x=by_band["pct"], y=by_band["faixa"], mode="markers",
    error_x=dict(type="data", symmetric=False, array=by_band["hi"] - by_band["pct"],
                 arrayminus=by_band["pct"] - by_band["lo"])))
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(title="Efeito ajustado de usar IA no salário, por experiência (%)",
                  xaxis_title="variação % (IC 95%)", yaxis=dict(categoryorder="array", categoryarray=ORDEM_EXP[::-1]))
fig.show(); save_fig(fig, "g13_efeito_ia_por_faixa")

WindowsPath('figures/g13_efeito_ia_por_faixa.html')

## 3. Q3: que fatores se associam ao salário?
Comparamos três modelos no **mesmo conjunto de teste** (20% das linhas, nunca vistas no treino):
um *baseline* (prever sempre a mediana), a regressão linear e o XGBoost. Se o XGBoost não superar bem o baseline, não há nada a interpretar.

In [31]:
model, comp, (X_tr, X_te, y_tr, y_te) = M.train_and_compare(X, y)
comp.to_csv("reports/q3_metricas.csv")
print("árvores usadas:", model.best_iteration + 1)
comp

árvores usadas: 718


,r2,mae_log,erro_mediano_%
baseline (mediana),-0.034,0.705,47.3
regressão linear,0.583,0.426,28.1
XGBoost,0.611,0.401,26.1


In [32]:
pred = model.predict(X_te)
s = np.random.default_rng(0).choice(len(y_te), size=min(3000, len(y_te)), replace=False)
fig = px.scatter(x=np.exp(y_te.to_numpy()[s]), y=np.exp(pred[s]), opacity=0.35, log_x=True, log_y=True,
                 labels={"x": "Salário real (USD)", "y": "Salário previsto (USD)"},
                 title="XGBoost: salário real vs previsto (teste, 3000 pontos)")
lim = [np.exp(y_te.min()), np.exp(y_te.max())]
fig.add_trace(go.Scatter(x=lim, y=lim, mode="lines", line=dict(dash="dash", color="gray"), name="previsão perfeita"))
fig.show(); save_fig(fig, "g14_real_vs_previsto")

WindowsPath('figures/g14_real_vs_previsto.html')

### Teste de robustez: treinar com 2024, prever 2025
Sem a variável `ano`. O **viés médio** é quanto os salários de 2025 diferem do que o modelo de 2024 prevê, dadas as mesmas características (experiência, país, função...).

In [33]:
temporal = M.temporal_check(X, y, d)
temporal

{'r2': 0.601, 'mae_log': 0.39, 'erro_mediano_%': 25.2, 'vies_medio_%': 0.6}

### O que o modelo aprendeu (SHAP)
SHAP reparte cada previsão pelo contributo de cada variável. Calculamos numa amostra de 4000 linhas do teste.

In [34]:
samp = X_te.sample(min(4000, len(X_te)), random_state=0)
sv = M.shap_values(model, samp)

grp = M.shap_by_group(sv, groups)
grp.to_csv("reports/q3_shap_grupos.csv")
grp

,importancia_%
pais,50.5
linguagem,15.0
experiencia,14.0
org,6.9
educacao,4.6
remoto,4.4
funcao,3.6
ia,0.8
ano,0.2


In [35]:
fig = px.bar(grp.reset_index(), x="importancia_%", y="index", orientation="h", text="importancia_%",
             title="Importância por grupo de variáveis (SHAP, % do total)",
             labels={"importancia_%": "% da importância", "index": ""})
fig.update_yaxes(autorange="reversed"); fig.update_traces(textposition="outside")
fig.show(); save_fig(fig, "g15_shap_grupos")

WindowsPath('figures/g15_shap_grupos.html')

In [36]:
lang = M.shap_binary_effect(sv, samp, "lang__")
lang.to_csv("reports/q3_shap_linguagens.csv", index=False)
fig = px.bar(lang.sort_values("efeito_%"), x="efeito_%", y="variavel", orientation="h",
             hover_data=["n"], text="efeito_%",
             title="Efeito médio no salário de usar cada linguagem, segundo o modelo (%)",
             labels={"efeito_%": "variação % no salário", "variavel": ""})
fig.update_layout(height=650)
fig.show(); save_fig(fig, "g16_shap_linguagens")
lang

,variavel,n,efeito_%
0,Go,638,8.1
1,Rust,498,7.6
2,Ruby,244,5.7
3,Swift,171,5.4
4,Bash/Shell (all shells),1738,3.8
5,TypeScript,1780,3.6
6,Kotlin,418,2.7
7,Groovy,177,1.9
8,C++,749,1.7
9,Python,2071,0.8


In [37]:
roles = M.shap_binary_effect(sv, samp, "funcao__")
roles.to_csv("reports/q3_shap_funcoes.csv", index=False)
roles

,variavel,n,efeito_%
0,Engineering manager,132,21.9
1,"Architect, software or solutions",130,6.7
2,"Developer, back-end",799,3.6
3,Data engineer,83,1.0
4,"Developer, embedded applications or devices",145,0.2
5,"Developer, full-stack",1354,-1.5
6,"Developer, mobile",123,-3.7
7,"Developer, front-end",194,-4.5
8,"Developer, desktop or enterprise applications",182,-4.8
9,Academic researcher,82,-27.9


## 4. Q4: a confiança na IA mudou entre 2024 e 2025?
Só **utilizadores** (a base é comparável). `Mann-Whitney` testa se as distribuições diferem; o **delta de Cliff** mede o tamanho do efeito (|δ| < 0.147 desprezável, < 0.33 pequeno, < 0.474 médio). Com milhares de respostas o valor-p dá quase sempre "significativo": o que interessa é o tamanho do efeito.

A coluna *ajustado* repete a comparação controlando experiência e país.

In [38]:
users = df[df["uses_ai"] == True].copy()
rows = {}
for nome, col in (("Sentimento (1-5)", "ai_sentiment_score"),
                  ("Confiança na precisão (1-5)", "ai_accuracy_score"),
                  ("Capacidade em tarefas complexas (1-5)", "ai_complexity_score")):
    rows[nome] = {**M.compare_years(users, col), **M.adjusted_year_effect(users, col)}
q4 = pd.DataFrame(rows).T
q4.to_csv("reports/q4_testes.csv")
q4

ModuleNotFoundError: No module named 'src.features'

In [ ]:
fig = px.bar(q4.reset_index(), x="index", y=["media_2024", "media_2025"], barmode="group",
             title="Médias (1-5) entre utilizadores de IA", labels={"index": "", "value": "média"})
fig.update_yaxes(range=[1, 5])
fig.show(); save_fig(fig, "g17_testes_q4")

## 5. Q5: que perfis de developers existem?
K-Means sobre experiência, uso de IA, 15 linguagens e 8 funções. **Decisão:** o número de clusters `k`. Vê a silhouette (quanto mais alta, mais separados), mas também se os perfis se **interpretam**. A silhouette tende a escolher o `k` mais pequeno; com variáveis binárias costuma ser baixa: os perfis são *tendências*, não grupos nítidos.

In [ ]:
d_c, Xc, L, R = M.cluster_matrix(df)
ks = M.choose_k(Xc)
ks

In [ ]:
fig = px.line(ks, x="k", y="silhouette", markers=True, title="Silhouette por número de clusters")
fig.show(); save_fig(fig, "g18_silhouette")

In [ ]:
K = int(ks.loc[ks["silhouette"].idxmax(), "k"])      # muda aqui se preferires outro k
print("k escolhido:", K)
from sklearn.cluster import KMeans
labels = KMeans(n_clusters=K, n_init=10, random_state=M.SEED).fit_predict(Xc)

perfis = M.describe_clusters(d_c, labels, L, R)
perfis.to_csv("reports/q5_clusters.csv", index=False)
perfis

In [ ]:
pc = M.pca_2d(Xc, labels)
fig = px.scatter(pc, x="pc1", y="pc2", color="cluster", opacity=0.5,
                 title=f"Perfis de developers (K-Means, k={K}) em 2 componentes principais")
fig.show(); save_fig(fig, "g19_clusters_pca")

fig = px.bar(perfis, x="cluster", y="pct_usa_ia", text="pct_usa_ia", title="Adoção de IA por perfil (%)",
             labels={"pct_usa_ia": "% que usa IA", "cluster": "perfil"})
fig.show(); save_fig(fig, "g20_clusters_ia")

## Guardar o modelo (para o dashboard do Dia 5)

In [ ]:
model.save_model("models/xgb_salary.json")
json.dump({"features": list(X.columns), "groups": groups}, open("models/features.json", "w"), ensure_ascii=False)
print("modelo e lista de variáveis guardados em models/")

## Conclusões do Dia 4 (preencher)

**Q1, adoção:** (bruta vs ajustada)
-

**Q2, salário e IA / salários subiram?:** (efeito de `uses_ai` e de `year_2025` no modelo 4; sinal por faixa)
-

**Q3, fatores do salário:** (importância por grupo; linguagens com efeito; qualidade do modelo vs baseline)
-

**Q4, confiança:** (diferença e tamanho do efeito)
-

**Q5, perfis:** (nomes dados a cada cluster)
-

**Limitações:**
- Associações, não causalidade; variáveis omitidas (empresa, setor, tipo de contrato).
- Salários em USD nominais e autodeclarados; amostra limitada a quem declarou salário.
- A silhouette baixa indica que os perfis são tendências.
- O SHAP descreve o modelo, não o mercado.